# Doğal Dil İşleme + Karar Bilimi (Natural Language Processing + Decision Science)

👩🏻‍🏫 Bu görevde şunları bir araya getireceğiz:
* 🗣 Doğal Dil İşleme (Natural Language Processing)
* 📊 Karar Bilimi (Decision Science)

🎯 Amaç, Olist üzerindeki ürünlerin ve satıcıların **olumsuz (kötü) yorumlarını** anlamaktır.

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
# Data Manipulation
import numpy as np
import pandas as pd
pd.set_option("display.max_columns",None)

# Machine Learning
from sklearn.pipeline import make_pipeline

# Language Processing
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
import nltk
import string
import unidecode as unidecode

# Vectorizers and NLP Models
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.decomposition import LatentDirichletAllocation

🕵🏻‍♂️ Olist’in CEO’su [Tiago Dalvi](https://www.linkedin.com/in/tiagodalvi/)’nin senden yorumları okuyup anlamanı istediğini hayal et.

- Müşteriler siparişlerini **1**, **2** veya **3** puanla değerlendirdiklerinde ne söylediler?
- En sık karşılaşılan olumsuz yorumlar neler?
    - En kötü puanlanan ürünler hakkında?
    - En kötü puanlanan satıcılar hakkında?


## (0) Kurulum 🔨

Öncelikle, Olist incelemeleriyle ilgili tüm bilgileri içeren DataFrame'i yükleyeceğiz!

In [3]:
df = pd.read_csv("https://d32aokrjazspmn.cloudfront.net/materials/olist_reviews.csv")

In [4]:
df.head()

,review_id,length_review,review_score,order_id,product_category_name,full_review
0,e64fb393e7b32834bb789ff8bb30750e,37,5,658677c97b385a9be170737859d3511b,ferramentas_jardim,Recebi bem antes do prazo estipulado.
1,f7c4243c7fe1938f181bec41a392bdeb,100,5,8e6bfb81e283fa7e4f11123a3fb894f1,esporte_lazer,Parabéns lojas lannister adorei comprar pela ...
2,8670d52e15e00043ae7de4c01cc2fe06,174,4,b9bf720beb4ab3728760088589c62129,eletroportateis,recomendo aparelho eficiente. no site a marca ...
3,4b49719c8a200003f700d3d986ea1a19,45,4,9d6f15f95d01e79bd1349cc208361f09,beleza_saude,"Mas um pouco ,travando...pelo valor ta Boa.\r\n"
4,3948b09f7c818e2d86c9a546758b2335,56,5,e51478e7e277a83743b6f9991dbfa3fb,informatica_acessorios,"Super recomendo Vendedor confiável, produto ok..."


## (2) Metin Temizleme (Text Cleaning)

🧹 `cleaning(sentence)` işlevini oluşturun ve yorumlara uygulayın. **NLTK'da Portekizce lemmatizer bulunmadığını unutmayın** (genellikle bulunmaz, ancak `nltk.stem.RSLPStemmer` kök bulucu vardır).

In [5]:
from nltk.stem import RSLPStemmer
nltk.download("rslp")
stemmer = RSLPStemmer()

[nltk_data] Downloading package rslp to /Users/iremtaskin/nltk_data...
[nltk_data]   Package rslp is already up-to-date!


In [6]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
import string

stop_words = set(stopwords.words('portuguese')) 
def cleaning(sentence):

    sentence = sentence.lower() 
    sentence = ''.join(char for char in sentence if not char.isdigit()) 
    for punctuation in string.punctuation:
        sentence = sentence.replace(punctuation, '') 
    
    tokenized_sentence = word_tokenize(sentence) 

    tokenized_sentence = [w for w in tokenized_sentence
                                  if not w in stop_words] 
    stemmed_sentence = [stemmer.stem(word)
              for word in tokenized_sentence] 
    decoded_sentence = [unidecode.unidecode(w) for w in stemmed_sentence] ## remove accents

    cleaned_sentence = ' '.join(decoded_sentence) ## join back into a string

    return cleaned_sentence 

In [7]:
df["full_review_cleaned"] = df["full_review"].apply(cleaning)
df["full_review_cleaned"]

0                               receb bem ant praz estipul
1        parabem loj lannist ador compr internet segur ...
2        recom aparelh efici sit marc aparelh impress d...
3                              pouc travandopel val ta boa
4           sup recom vend confi produt ok entreg ant praz
                               ...                        
36143                      produt receb acord compr realiz
36144    entreg dentr praz produt cheg cond perfeit sat...
36145    produt envi nf exist vend nf cert fic aguard e...
36146          excel mochil entreg sup rapid sup recom loj
36147          produt cheg ja devolv poi defeit segur carg
Name: full_review_cleaned, Length: 36148, dtype: object

## (3) Kötü yorumların analizi

### (3.1) Düşük inceleme puanlarına sahip veri kümesi

😱 1 ile 3 arasında puan alan yorumların oranı nedir? 

In [8]:
round(df["review_score"].value_counts(normalize = True),2)

review_score
5    0.57
4    0.16
1    0.13
3    0.09
2    0.05
Name: proportion, dtype: float64

🕵🏻‍♂️ Bu yorumlara odaklanalım...

In [9]:
threshold = 3
bad = df["review_score"] <= threshold
df = df[bad]
df.shape

(9658, 7)

In [10]:
df.head()

,review_id,length_review,review_score,order_id,product_category_name,full_review,full_review_cleaned
14,e233e51d11511bf30e568c76360ace52,97,1,548df2c6e5f089574614894bca78acf5,eletronicos,recebi somente 1 controle Midea Split ESTILO....,receb soment control mide split estil falt con...
23,8b230a1373c6dc4bd867099fda1d7039,60,3,071251fe3b3493294536f03737a8a679,ferramentas_jardim,Eu comprei duas unidades e só recebi uma e ag...,compr dua unidad receb agor fac
24,cb2fc3e5711b5ae85e0491ee18af63ed,72,3,34e6d418f368f8079ae152bc178bc66a,beleza_saude,"Produto bom, porém o que veio para mim não co...",produt bom porem vei mim condiz fot anunci
25,60c714ed14cef913944a3147094a4742,36,1,9ac05114800f02bfaa783bd76842dbe2,moveis_decoracao,"Produto muito inferior, mal acabado.",produt inferi mal acab
28,0bd4dcc4f6c4621baf37f73495cad8c4,16,3,a11cd01ac67beef7e8bf09740d8a35c1,esporte_lazer,Entrega no prazo,entreg praz


### (3.2) Vektörleştirme

🔡 ➡️ 🔢 Metinlerini vektörleştir.

- **Bigram**’leri (iki kelimelik ifadeler) mutlaka hesaba kat.
- Çok sık geçen kelimeleri çıkarmak için `max_df = 0.75` ayarla.
- Spoiler uyarısı: Sonunda **20.000+** kelimeye ulaşacaksın…  
  Bu challenge için sadece `max_features = 5000` ile sınırla.

In [11]:
vectorizer = TfidfVectorizer(max_df = 0.75, max_features = 5000, ngram_range=(1,2))
vectorized_reviews = pd.DataFrame(vectorizer.fit_transform(df["full_review_cleaned"]).toarray(),
                                 columns = vectorizer.get_feature_names_out())

print(f" vectorized_reviews.shape = {vectorized_reviews.shape}")

 vectorized_reviews.shape = (9658, 5000)


### (3.3) LDA

🕵🏻‍♂️ LDA'yı uyarlayın:
- `n_components = 3` seçin
- *.transform()* ile Konuların Belge Karışımını gösterin
- *.components_* ile Konu Karışımını gösterin

In [12]:
n_components = 3

lda = LatentDirichletAllocation(n_components = n_components)
lda.fit(vectorized_reviews)

LatentDirichletAllocation(n_components=3)

#### Belge Karışımı (Konular için)

In [13]:
document_mixture = lda.transform(vectorized_reviews)
document_mixture.shape

(9658, 3)

👉 Her inceleme için en önemli konuyu rapor edelim

In [14]:
round(pd.DataFrame(document_mixture,
                   columns = [f"Topic {i+1}" for i in range(n_components)])
      ,2)

,Topic 1,Topic 2,Topic 3
0,0.08,0.70,0.22
1,0.08,0.08,0.84
2,0.67,0.10,0.23
3,0.79,0.10,0.10
4,0.13,0.74,0.13
...,...,...,...
9653,0.08,0.12,0.81
9654,0.77,0.11,0.11
9655,0.57,0.35,0.08
9656,0.35,0.10,0.56


In [15]:
df["most_important_topic"] = np.argmax(document_mixture, axis = 1)
df.head()

,review_id,length_review,review_score,order_id,product_category_name,full_review,full_review_cleaned,most_important_topic
14,e233e51d11511bf30e568c76360ace52,97,1,548df2c6e5f089574614894bca78acf5,eletronicos,recebi somente 1 controle Midea Split ESTILO....,receb soment control mide split estil falt con...,1
23,8b230a1373c6dc4bd867099fda1d7039,60,3,071251fe3b3493294536f03737a8a679,ferramentas_jardim,Eu comprei duas unidades e só recebi uma e ag...,compr dua unidad receb agor fac,2
24,cb2fc3e5711b5ae85e0491ee18af63ed,72,3,34e6d418f368f8079ae152bc178bc66a,beleza_saude,"Produto bom, porém o que veio para mim não co...",produt bom porem vei mim condiz fot anunci,0
25,60c714ed14cef913944a3147094a4742,36,1,9ac05114800f02bfaa783bd76842dbe2,moveis_decoracao,"Produto muito inferior, mal acabado.",produt inferi mal acab,0
28,0bd4dcc4f6c4621baf37f73495cad8c4,16,3,a11cd01ac67beef7e8bf09740d8a35c1,esporte_lazer,Entrega no prazo,entreg praz,1


#### Konu Karışımı (Kelimeler için)

In [16]:
topic_mixture = pd.DataFrame(lda.components_,
                             columns = vectorizer.get_feature_names_out())
topic_mixture.shape

(3, 5000)

In [17]:
topic_mixture

abaix      abert   aborrec        abr  abr caix  abr cham  abr embal  \
0  1.910423  13.157014  2.702076  16.596966  1.502467  1.807532   3.564519   
1  2.701737   0.487074  0.345626   4.655680  0.365036  0.335397   0.350774   
2  0.369691   0.368800  0.337503   3.221217  1.318141  0.410559   0.503255   

   abr fech  abr reclam     abril    absurd  absurd entreg      abus  \
0  0.345718    5.076512  1.858519  4.854202       1.020937  0.389237   
1  1.756330    0.335816  0.336582  4.603081       1.500772  0.626424   
2  0.343138    1.238046  0.360470  3.935722       0.335553  0.905997   

        aca       acab  acab compr  acab deix  acab fic  acab produt  \
0  1.328969  23.927398    0.348580   2.659621  1.076500     1.141686   
1  0.334285  10.524077    1.389981   0.394049  1.208789     1.109158   
2  0.337509   0.363160    0.661290   0.335064  0.343654     0.337403   

   acab receb  acab recom  acab ruim     aceit     acend     acert     acess  \
0    0.447185    1.675803   3.965200  5.099767  0.365183  1.488910  2.816830   
1    0.362616    0.355572   0.334809  2.387891  2.644076  0.362804  0.411047   
2    2.270632    0.335963   0.335883  0.345227  0.342336  0.818758  0.471368   

   acessori        ach  ach absurd   ach bem   ach car  ach cord  ach demor  \
0  0.872830  42.866884    1.105206  1.754109  0.351042  0.333703   0.359401   
1  2.586951  28.999133    0.597576  0.761323  1.626521  1.128880   5.335988   
2  1.502466   1.555678    0.410414  0.358893  0.335376  0.333803   0.342053   

    ach dev  ach entreg  ach estranh  ach fornec  ach fret   ach loj  \
0  3.422408    0.364958     1.716569    0.333891  2.649326  0.401229   
1  1.935887    2.293465     0.451872    1.241932  0.349936  0.501491   
2  0.351858    0.438860     0.339122    0.333918  0.336916  1.976079   

   ach mant  ach mater  ach melhor  ach pequen  ach pess  ach pouc  ach praz  \
0  0.341732   3.078479    1.770311    0.343769  0.388964  2.599321  2.703461   
1  0.335939   0.377341    0.345191    4.781720  1.951866  2.402418  0.878205   
2  2.115892   0.335757    0.651510    0.343095  0.374417  0.339776  0.337712   

   ach prec  ach produt  ach qual   ach vei      acid      acim     acion  \
0  2.602897    8.545743  2.212973  0.353884  0.339073  0.356311  1.385860   
1  0.342419    1.467887  0.345933  1.743519  1.826389  0.762019  1.841994   
2  0.334939    0.342211  0.367434  0.808645  0.336287  1.810261  0.342710   

        aco  acompanh  acompanh ped  acompanh produt  acondicion   acontec  \
0  1.439706  4.153708      0.348321         2.117804    0.544306  3.981705   
1  0.365512  3.377671      0.337098         0.523545    2.590925  7.489236   
2  1.226417  2.294779      2.347170         0.339708    0.336804  8.547607   

   acontec compr  acontec produt     acord  acord descr   acostum  \
0       1.012386        0.355689  3.789505     0.501303  0.353143   
1       0.366368        2.902373  7.667892     1.384465  0.956106   
2       0.820715        0.349909  0.360603     0.376685  1.786089   

   acostum compr   acredit     acril  acril fic      acus     adapt     adequ  \
0       0.346852  3.202159  0.343307   0.334384  0.350656  1.956474  2.290001   
1       0.400757  8.780687  2.740956   0.360574  1.295444  2.106466  3.366017   
2       1.543329  0.732038  2.501658   1.879035  2.051040  1.836702  0.581206   

       ader      ades  ades pared  ades pra    adiant      ador    adquir  \
0  0.334227  3.188764    0.334781  0.333707  1.752234  8.597277  3.773806   
1  2.221055  4.489653    0.353665  1.109433  0.339650  2.390888  8.431616   
2  0.336063  3.304424    1.931699  0.333773  1.127453  0.348488  3.763978   

   adquir consegu  adquir produt     adult      afet  afet ades       aff  \
0        1.682407       0.359597  1.108469  0.934972   0.333707  0.778271   
1        0.334256       3.290365  0.600577  1.759562   1.109433  0.951843   
2        0.334339       0.372752  0.730879  0.337815   0.333773  0.348149   

     afinal     afirm    

#### Konular

🎁 Size bazı yardımcı fonksiyonlar sağladık:
- `topic_word`: Tek bir konu (topic) için en önemli kelimeleri ve ağırlıklarını döndürür
- `print_topics`: LDA tarafından bulunan farklı konuları, en önemli kelimeleriyle birlikte yazdırır

In [18]:
def topic_word(vectorizer, model, topic, topwords, with_weights = True):
    topwords_indexes = topic.argsort()[:-topwords - 1:-1]
    if with_weights == True:
        topwords = [(vectorizer.get_feature_names_out()[i], round(topic[i],2)) for i in topwords_indexes]
    if with_weights == False:
        topwords = [vectorizer.get_feature_names_out()[i] for i in topwords_indexes]
    return topwords

In [19]:
def print_topics(vectorizer, model, topwords):
    for idx, topic in enumerate(model.components_):
        print("-"*20)
        print("Topic %d:" % (idx))
        print(topic_word(vectorizer, model, topic, topwords))


🕵🏻‍♂️ Konuları en çok kullanılan kelimelerle birlikte yazdırın:

In [20]:
print_topics(vectorizer, lda, topwords = 5)

--------------------
Topic 0:
[('produt', 215.82), ('vei', 148.05), ('compr', 116.28), ('outr', 93.84), ('err', 86.61)]
--------------------
Topic 1:
[('produt', 166.69), ('entreg', 151.75), ('praz', 117.86), ('qual', 105.91), ('recom', 94.54)]
--------------------
Topic 2:
[('receb', 269.4), ('bom', 232.16), ('compr', 146.52), ('produt', 139.26), ('receb produt', 116.53)]


🇧🇷 Burada biraz Brezilya Portekizcesi kelimeler var:
- _cadeiras = chairs_
- _produto = product_
- _recomendo = recommend (não recomendo == not recommend)_
- _bom = good_
- _comprei = bought_
- _veio = came_
- _errado = wrong_
- _gostaria = I would like to..._
- _duas = two_
- _nao = not_
- _entregue = delivered_
- _pecas = part_
- _ainda = yet_
- _recebi = received_

👉 Bir konuyla ilişkili en popüler kelimeleri göster

In [21]:
topic_word_mixture = [topic_word(vectorizer, lda, topic, topwords = 5, with_weights = False)
                      for topic in lda.components_]
topic_word_mixture

[['produt', 'vei', 'compr', 'outr', 'err'],
 ['produt', 'entreg', 'praz', 'qual', 'recom'],
 ['receb', 'bom', 'compr', 'produt', 'receb produt']]

In [22]:
df["most_important_words"] = df["most_important_topic"].apply(lambda i: topic_word_mixture[i])

In [23]:
df[["review_id",
        "review_score",
        "product_category_name",
        "full_review_cleaned",
        "most_important_topic",
        "most_important_words"]
      ].head()

,review_id,review_score,product_category_name,full_review_cleaned,most_important_topic,most_important_words
14,e233e51d11511bf30e568c76360ace52,1,eletronicos,receb soment control mide split estil falt con...,1,"[produt, entreg, praz, qual, recom]"
23,8b230a1373c6dc4bd867099fda1d7039,3,ferramentas_jardim,compr dua unidad receb agor fac,2,"[receb, bom, compr, produt, receb produt]"
24,cb2fc3e5711b5ae85e0491ee18af63ed,3,beleza_saude,produt bom porem vei mim condiz fot anunci,0,"[produt, vei, compr, outr, err]"
25,60c714ed14cef913944a3147094a4742,1,moveis_decoracao,produt inferi mal acab,0,"[produt, vei, compr, outr, err]"
28,0bd4dcc4f6c4621baf37f73495cad8c4,3,esporte_lazer,entreg praz,1,"[produt, entreg, praz, qual, recom]"


## (3.4) Pipeline Tf-Idf ve LDA

In [24]:
from sklearn import set_config
set_config("diagram")

🔨 Önceki Vectorizer ve LDA'yı birbirine bağlayan bir Pipeline oluşturun.

Temizlenmiş metinlere uyarlayın.

In [25]:
%%time
max_df = 0.75
max_features = 5000
ngram_range = (1,2)

pipeline = make_pipeline(
    TfidfVectorizer(max_df = max_df,
                    max_features = max_features,
                    ngram_range = ngram_range),
    LatentDirichletAllocation(n_components = n_components)
)

pipeline.fit(df["full_review_cleaned"])

CPU times: user 3.82 s, sys: 9.37 ms, total: 3.83 s
Wall time: 3.83 s


Pipeline(steps=[('tfidfvectorizer',
                 TfidfVectorizer(max_df=0.75, max_features=5000,
                                 ngram_range=(1, 2))),
                ('latentdirichletallocation',
                 LatentDirichletAllocation(n_components=3))])

💡 `pipeline.components_` ile bileşenlere erişmeye çalışırsanız, Pipeline'da `components_` olmadığı için bu YÜRÜMEZ. Ancak, LDA'ya erişmek için `pipeline._final_estimator` kullanabilirsiniz. Ve buradan konulara erişebilirsiniz!

In [26]:
pipeline._final_estimator

LatentDirichletAllocation(n_components=3)

In [27]:
pipeline._final_estimator.components_

array([[ 0.35759579, 12.28339733,  1.06751768, ...,  0.34837135,
         7.19153357,  6.18558287],
       [ 4.24694265,  0.35206483,  1.11642943, ...,  0.33460285,
         1.68909558,  0.36009666],
       [ 0.37731193,  1.37742513,  1.20125833, ...,  2.80612541,
         0.36973306,  0.41274588]])

Pipeline ile **Belge Karışımı**:

In [28]:
document_mixture = pipeline.transform(df["full_review_cleaned"])
document_mixture.shape

(9658, 3)

Pipeline ile **Konu Karışımı**:

In [29]:
topic_mixture = pd.DataFrame(pipeline._final_estimator.components_)
topic_mixture.shape

(3, 5000)

In [30]:
topic_mixture

0          1         2          3         4         5         6     \
0  0.357596  12.283397  1.067518  12.782037  1.764961  0.374001  3.527169   
1  4.246943   0.352065  1.116429   6.452152  1.074689  1.831501  0.470317   
2  0.377312   1.377425  1.201258   5.239674  0.345994  0.347986  0.421063   

       7         8         9         10        11        12        13    \
0  1.766152  0.349286  1.672575  0.394056  0.334082  0.350513  1.050629   
1  0.336591  3.803901  0.546515  3.690212  0.340304  1.236258  0.364988   
2  0.342444  2.497187  0.336481  9.308737  2.182877  0.334887  0.585146   

        14        15        16        17        18        19        20    \
0  28.498083  1.662338  2.715419  1.731852  0.503652  2.122058  0.742605   
1   1.201358  0.364981  0.334375  0.334965  1.681860  0.596894  0.361314   
2   5.115194  0.372532  0.338940  0.562126  0.402736  0.361480  1.263419   

       21        22        23        24        25        26         27    \
0  0.360869  1.511366  2.658477  0.843714  1.093299  0.427450  17.456942   
1  0.336054  1.677989  0.341116  1.484207  0.675595  0.436275  21.880365   
2  3.938968  4.643530  0.352002  0.342551  1.930351  4.098523  34.084387   

       28        29        30        31        32        33        34    \
0  0.337760  1.685875  0.421821  0.333779  0.340227  1.008195  0.335082   
1  1.216429  0.335254  1.547459  0.333783  5.078858  1.274381  1.416078   
2  0.559007  0.853196  0.343660  1.128824  0.618356  3.427577  1.346122   

       35        36        37        38        39        40        41    \
0  1.299559  0.333839  0.400493  0.337024  0.334243  0.359053  0.373645   
1  0.335600  0.333935  0.544152  2.178946  2.124974  0.337337  1.910343   
2  0.872405  1.241966  2.391533  0.362828  0.334346  3.095187  0.483024   

       42        43        44        45        46        47        48    \
0  0.357675  1.001079  0.354440  0.338256  0.334508  1.697663  2.253847   
1  4.753691  0.344176  4.570947  3.110602  0.338866  0.542597  0.336432   
2  0.357218  1.369992  0.416128  0.470520  2.606881  8.115581  0.336061   

       49        50        51        52        53        54        55    \
0  0.341654  1.779560  0.991379  2.084251  1.057733  3.464206  2.028912   
1  1.387997  0.336253  0.821407  0.772048  0.412084  1.764859  0.659890   
2  1.176397  0.385936  1.115805  0.714265  1.561819  4.597093  0.343787   

       56        57        58        59        60        61        62    \
0  0.345738  2.797844  3.403993  0.351938  0.351478  3.473257  0.835281   
1  0.348757  0.334613  6.985862  1.506268  0.344704  3.743953  0.342899   
2  2.286561  0.339578  9.628693  0.341262  2.911789  4.600789  1.084273   

       63        64        65        66        67        68        69    \
0  0.636572  0.337321  6.650861  0.446543  0.342901  0.500250  2.049528   
1  2.121334  1.618883  5.558058  0.336798  0.334190  1.760421  0.410062   
2  0.337432  0.334734  0.505965  4.802580  1.896902  1.436469  3.440051   

       70        71        72        73        74        75         76    \
0  1.446046  0.341027  7.886878  0.432437  1.108123  0.713459  10.602678   
1  0.885971  0.335241  2.137557  1.827679  0.333790  2.022008   0.358132   
2  3.905208  2.215076  0.958406  0.360029  0.335001  0.483870   0.375843   

       77        78        79        80        81        82        83    \
0  3.275585  0.334796  0.624676  0.384668  1.339223  1.108123  0.732262   
1  7.056644  1.673884  0.772799  1.380470  0.976290  0.333790  0.943340   
2  5.637172  0.342322  2.625239  0.674787  0.716836  0.335001  0.402661   

       84        85        86        87        88        89         90    \
0  0.337493  0.946296  0.336852  0.336609  0.338801  0.336152  14.265126   
1  1.420288  1.093065  0.341048  0.342158  3.390452  1.940662  27.501768   
2  0.717779  0.498328  8.527788  6.113608  2.230934  0.345999  16.136661   

       91        92        93        94        95        96        97 

## (4) 🎁 Ürün Kategorileri

### (4.1) Ürün kategorilerine göre gruplandırma

📈 Veri kümesini `product_category_name` ile gruplandırın ve performanslarını inceleyin.

In [31]:
# Performansa göre ürün kategorileri - sayı, ortalama, medyan ve standart sapmaya bakın
product_categories = df.groupby(by = 'product_category_name').agg({
        'review_score': ["count", "mean", "median", "std"]
    })

# Analiz için belirli bir süreden daha az satılan ürünleri kaldırma
cutoff = 50
product_categories = product_categories[product_categories[("review_score", "count")] > cutoff]

# Ürün kategorilerini performansa göre sıralama
product_categories = product_categories.sort_values(by = [('review_score', 'mean'),
                                                          ('review_score', 'std')],
                                                    ascending = [False, True])
product_categories

review_score                           
                                         count      mean median       std
product_category_name                                                    
consoles_games                              92  1.956522    2.0  0.924787
fashion_bolsas_e_acessorios                155  1.948387    2.0  0.917319
malas_acessorios                            66  1.924242    2.0  0.949666
papelaria                                  155  1.922581    2.0  0.922556
casa_construcao                             66  1.909091    2.0  0.836242
utilidades_domesticas                      586  1.889078    2.0  0.888473
cool_stuff                                 298  1.885906    2.0  0.906618
relogios_presentes                         589  1.881154    2.0  0.905088
pet_shop                                   147  1.857143    2.0  0.883641
moveis_escritorio                          261  1.854406    1.0  0.920843
cama_mesa_banho                           1221  1.850942    2.0  0.890235
esporte_lazer                              626  1.846645    2.0  0.899136
casa_conforto                               57  1.842105    2.0  0.902169
beleza_saude                               716  1.840782    1.0  0.908275
ferramentas_jardim                         318  1.817610    1.0  0.904635
telefonia                                  488  1.815574    2.0  0.875636
moveis_decoracao                           743  1.802153    1.0  0.888620
brinquedos                                 298  1.788591    1.0  0.890808
bebes                                      244  1.786885    1.0  0.877046
eletronicos                                248  1.778226    1.0  0.883502
construcao_ferramentas_construcao           71  1.774648    1.0  0.881511
automotivo                                 370  1.770270    1.0  0.870275
perfumaria                                 235  1.723404    1.0  0.879543
informatica_acessorios                     797  1.708908    1.0  0.867089
eletrodomesticos                            84  1.630952    1.0  0.875087
eletroportateis                             54  1.518519    1.0  0.770708

### (4.2) En kötü ürün kategorileri

👎 *Ortalama değerlendirme puanı* açısından en kötü beş kategoriyi `worst_products` adlı bir değişkene kaydedin.

In [32]:
worst_products = product_categories.tail(5).sort_values(by = [("review_score", "count")],
                                                       ascending = False)
worst_products

review_score                           
                              count      mean median       std
product_category_name                                         
informatica_acessorios          797  1.708908    1.0  0.867089
automotivo                      370  1.770270    1.0  0.870275
perfumaria                      235  1.723404    1.0  0.879543
eletrodomesticos                 84  1.630952    1.0  0.875087
eletroportateis                  54  1.518519    1.0  0.770708

👇 Yalnızca `worst_products` öğelerini içeren bir `worst_products_review` DataFrame oluşturun.

In [33]:
worst_products_reviews = df[df.product_category_name.isin(worst_products.index)]
worst_products_reviews[["review_id",
                        "review_score",
                        "product_category_name",
                        "full_review_cleaned",
                        "most_important_topic",
                        "most_important_words"]
      ]

,review_id,review_score,product_category_name,full_review_cleaned,most_important_topic,most_important_words
34,ff722b4c68783a4459a3adb9bb4e1d0d,3,informatica_acessorios,produt cheg pc consegu reconhec port usb,0,"[produt, vei, compr, outr, err]"
57,5f938e5f5f2e9a75710b54feeb9ea610,1,eletrodomesticos,medi pec serv,1,"[produt, entreg, praz, qual, recom]"
71,6b341682ab39af9fa00d72d7388c903b,3,automotivo,not compr produt corret cap intern outr,1,"[produt, entreg, praz, qual, recom]"
88,b736ff4204044e49e39062584d06fa74,1,eletroportateis,loj anunc produt entreg outr,0,"[produt, vei, compr, outr, err]"
100,6f885e2fd69412264d9e74e805175d53,1,informatica_acessorios,receb bem rapid porem vei outr produt ped mode...,0,"[produt, vei, compr, outr, err]"
...,...,...,...,...,...,...
36049,e9f0bde9a98ff79305964e9033ed8cd2,1,informatica_acessorios,produt cheg falt pess cond,1,"[produt, entreg, praz, qual, recom]"
36054,44380245bf1a49875542ea0e7f216986,1,informatica_acessorios,receb produt aind assim receb produt avali posi,2,"[receb, bom, compr, produt, receb produt]"
36095,970ae7c9df61910a0a30c38670db7ae5,3,informatica_acessorios,probl past multius pra notebook ped cor vermel...,0,"[produt, vei, compr, outr, err]"
36100,504403288b387de1c95a193909ba5dfd,3,perfumaria,cheg bem rapid produt razoa,1,"[produt, entreg, praz, qual, recom]"


### (4.3). En kötü ürünler için konular

❓ En kötü ürünlerin konuları nelerdir? ❓

In [34]:
worst_products_reviews["most_important_topic"].value_counts()

most_important_topic
0    552
1    518
2    470
Name: count, dtype: int64

In [35]:
bad_frequency = list(worst_products_reviews["most_important_topic"].value_counts().index)
bad_frequency

[0, 1, 2]

In [36]:
[topic_word_mixture[i] for i in bad_frequency]

[['produt', 'vei', 'compr', 'outr', 'err'],
 ['produt', 'entreg', 'praz', 'qual', 'recom'],
 ['receb', 'bom', 'compr', 'produt', 'receb produt']]

## (5) 🎁 Satıcılar...

* En kötü satıcılar tarafından ne tür ürünler satıldı?
* En kötü satıcılar için başlıca yorumlar nelerdir?

### (5.1) En kötü satıcılar

In [37]:
from olist.seller import Seller
sellers = Seller().get_training_data()
sellers.columns

Index(['seller_id', 'seller_city', 'seller_state', 'delay_to_carrier',
       'wait_time', 'date_first_sale', 'date_last_sale', 'months_on_olist',
       'n_orders', 'quantity', 'quantity_per_order', 'sales',
       'share_of_five_stars', 'share_of_one_stars', 'review_score'],
      dtype='object')

👇 En kötü satan 10 ürünü seçin ve bunları `worst_sellers` adlı bir değişkene kaydedin.

In [39]:
worst_sellers = sellers[["seller_id", "review_score"]].sort_values(
    by="review_score", ascending=True).head(10)
worst_sellers

,seller_id,review_score
2014,1fa2d3def6adfa70e58c276bb64fe5bb,1.0
1444,c87abc38c8ed3240861729e1aeadf221,1.0
2224,f524ad65d7e0f1daab730ef2d2e86196,1.0
864,1f7dfad2cb384ea4d2d7e1ffbd78c407,1.0
2254,c97aa4ee7420f937da13b7f9e2228b99,1.0
1396,9bf11dfc0bec77e5a23028043c3c5a8f,1.0
241,781a66c0d89e30814d49fe8b5e14cd07,1.0
1375,da2782c804606d2a5d8e1760dbb3e7ec,1.0
2215,fc6295add6f51a0936407ead70c1001d,1.0
2320,010da0602d7774602cd1b3f5fb7b709e,1.0


### (5.2) En kötü satıcılar tarafından satılan ürünler

In [40]:
from olist.product import Product	
products = Product().get_training_data() [["product_id", "category"]]
products

,product_id,category
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumery
1,3aa071139cb16b67ca9e5dea641aaa2f,art
2,96bd76ec8810374ed1b65e291975717f,sports_leisure
3,cef67bcfe19066a932b7673e239eb23d,baby
4,9dc1a7de274444849c219cff195d0b71,housewares
...,...,...
31479,a0b7d5a992ccda646f2d34e418fff5a0,furniture_decor
31480,bf4538d88321d0fd4412a93c974510e6,construction_tools_lights
31481,9a7c6041fa9592d9d9ef6cfe62a71f8c,bed_bath_table
31482,83808703fc0706a22e264b9d75f04a2e,computers_accessories


❓ En kötü satıcılar tarafından satılan ürün türleri nelerdir? ❓

In [41]:
from olist.data import Olist 
data = Olist().get_data()	
sellers_product_category = data["order_items"].merge(products,
                                             on = "product_id", how = "left")[["seller_id", "category"]]

sellers_product_category

,seller_id,category
0,48436dade18ac8b2bce089ec2a041202,cool_stuff
1,dd7ddc04e1b6c2c614352b383efe2d36,pet_shop
2,5b51032eddd242adc84c38acab88f23d,furniture_decor
3,9d7a1d34a5052409006425275ba1c2b4,perfumery
4,df560393f3a51e74553ab94004ba5c87,garden_tools
...,...,...
112645,b8bc237ba3788b23da09c0f1f3a3288c,housewares
112646,f3c38ab652836d21de61fb8314b69182,computers_accessories
112647,c3cfdc648177fdbbbb35635a37472c53,sports_leisure
112648,2b3e4a2a3ea8e01938cabda2a3e5cc79,computers_accessories


In [42]:
sellers_product_category.groupby("seller_id").count()

,category
seller_id,
0015a82c2db000af6aaaf3ae2ecb0532,3
001cca7ae9ae17fb1caed9dfb1094831,239
001e6ad469a905060d959994f1b41e4f,0
002100f778ceb8431b7a1020ff7ab48f,55
003554e2dce176b5555353e4f3555ac8,0
...,...
ffcfefa19b08742c5d315f2791395ee5,0
ffdd9f82b9a447f6f8d4b91554cc7dd3,20
ffeee66ac5d5a62fe688b9d26f83f534,14


### (5.3) En kötü satıcılar için kategoriler ve konular

🎁 İşte bazı kullanışlı işlevler:
- Bir satıcı tarafından satılan ürün kategorilerini göstermek için `focus_seller(seller_id)`
- Bir satıcı için en sık kullanılan konuların en popüler kelimelerini göstermek için `bad_reviews_seller`

In [43]:
def focus_seller(seller_id):
    return sellers_product_category[sellers_product_category.seller_id == seller_id].value_counts()

In [44]:
bad_reviews_sellers = worst_products_reviews.merge(data["order_items"])
bad_reviews_sellers.head(3)

,review_id,length_review,review_score,order_id,product_category_name,full_review,full_review_cleaned,most_important_topic,most_important_words,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,ff722b4c68783a4459a3adb9bb4e1d0d,66,3,b8702f50a689621ab21ef200b55a5b8d,informatica_acessorios,"Produto chegou, mas meu PC não conseguiu reco...",produt cheg pc consegu reconhec port usb,0,"[produt, vei, compr, outr, err]",1,e76438822c17316d276c27e52e588931,5656537e588803a555b8eb41f07a944b,2018-01-10 22:07:28,24.89,17.63
1,5f938e5f5f2e9a75710b54feeb9ea610,17,1,d9ff0185a300435402f043124f530274,eletrodomesticos,Médio A peça não serviu,medi pec serv,1,"[produt, entreg, praz, qual, recom]",1,082175ee534338e4fd967076a6f848e5,d91fb3b7d041e83b64a00a3edfb37e4f,2018-05-10 16:54:16,52.00,18.24
2,6b341682ab39af9fa00d72d7388c903b,66,3,fd65651f855608ad1aa52ff4a741c31a,automotivo,Nota3 Comprar um produto correto na capa mas i...,not compr produt corret cap intern outr,1,"[produt, entreg, praz, qual, recom]",1,1dce3fd38c13a3e74fe5000f33858442,72c5da29406b4234927b81855e7b64f6,2018-05-07 05:30:57,229.99,19.49


In [45]:
def bad_reviews_seller(bad_reviews_sellers, seller_id):
    mask = (bad_reviews_sellers.seller_id == seller_id)
    temp = bad_reviews_sellers[mask]
    if len(temp) > 0: # satıcı kötü yorumlar veri çerçevesinde görünüyorsa
        most_frequent_topic_seller = list(temp.most_important_topic.value_counts().head(1).index)[0]
        return topic_word_mixture[most_frequent_topic_seller]

❓Bu en az satan ürünlerin her biri için en sık kullanılan ürün kategorilerini ve kelimeleri gösterin ❓

In [46]:
for worst_seller in worst_sellers["seller_id"]:
    print("-"*50)
    print(f"Focusing on the seller #{worst_seller}...")
    print(focus_seller(worst_seller))
    print(bad_reviews_seller(bad_reviews_sellers, worst_seller))


--------------------------------------------------
Focusing on the seller #1fa2d3def6adfa70e58c276bb64fe5bb...
seller_id                         category  
1fa2d3def6adfa70e58c276bb64fe5bb  housewares    1
Name: count, dtype: int64
None
--------------------------------------------------
Focusing on the seller #c87abc38c8ed3240861729e1aeadf221...
seller_id                         category                
c87abc38c8ed3240861729e1aeadf221  fashion_bags_accessories    1
Name: count, dtype: int64
None
--------------------------------------------------
Focusing on the seller #f524ad65d7e0f1daab730ef2d2e86196...
seller_id                         category    
f524ad65d7e0f1daab730ef2d2e86196  garden_tools    2
Name: count, dtype: int64
None
--------------------------------------------------
Focusing on the seller #1f7dfad2cb384ea4d2d7e1ffbd78c407...
seller_id                         category
1f7dfad2cb384ea4d2d7e1ffbd78c407  baby        1
Name: count, dtype: int64
None
------------------------

🏁 Tebrikler. NLP'nin bazı temellerini (Ön İşleme + Vektörleştirme + NB/LDA) öğrendiniz ve bu yeni “uzmanlığı” Karar Bilimi ile birleştirdik.

💾 `git add / commit / push` yapmayı unutmayın.